# Prédire le prix des taxis à New York avec le Machine Learning (partie 2)

Notebook de l'article [Prédire le prix des taxis avec le Machine Learning : modélisation (partie 2)](https://moncoachdata.com/blog/prix-des-taxis-modelisation-predictive/) (MonCoachData). Il fait suite à la [partie 1 : l'analyse exploratoire](https://moncoachdata.com/blog/prediction-du-prix-des-taxis-analyse/).

**Données** : [New York City Taxi Fare Prediction (Kaggle)](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/data). Seul `train.csv` est utilisé. Dans Google Colab, la section 0 le télécharge ; en local, place `train.csv` (ou `train.csv.zip`) à côté du notebook.

**Versions** : LightGBM 4, XGBoost 2 ou 3, Scikit-learn 1.4 ou plus récent, Hyperopt.

**Temps de calcul** : sur 3 millions de lignes, le Random Forest prend plusieurs minutes et beaucoup de mémoire, et Hyperopt entraîne un modèle complet à chaque essai : environ 1 minute et demie par essai sur Colab, soit près de 2 heures pour 100 essais. Le notebook en fait 20 par défaut (`MAX_EVALS`), ce qui donne un résultat proche en une demi-heure. Pour tester vite, mets `NB_LIGNES = 500_000` et `MAX_EVALS = 10` dans la cellule de paramètres.

## 0. Récupérer les données (Google Colab uniquement)

Dans Colab, le fichier de la compétition n'est pas là. Cette cellule le télécharge depuis Kaggle ; en local, elle ne fait rien.

Avant de l'exécuter :
1. accepte le règlement de la [compétition](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/rules) avec ton compte Kaggle ;
2. crée un jeton API (Kaggle > Settings > API > *Create New Token*) : tu obtiens soit une chaîne de caractères à coller dans la cellule, soit un fichier `kaggle.json` (dans ce cas, laisse le champ vide : la cellule te demandera de le téléverser).

Ne partage jamais ton jeton ni ton `kaggle.json`.

Attention : `train.csv` pèse plusieurs Go, le téléchargement prend quelques minutes.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules


def fichier_valide(nom):
    # Le CSV est là, ou bien une archive zip complète (un téléchargement interrompu donne un zip invalide)
    archive = Path(nom + ".zip")
    return Path(nom).exists() or (archive.exists() and zipfile.is_zipfile(archive))


if EN_COLAB:
    archive = Path("train.csv.zip")
    if archive.exists() and not zipfile.is_zipfile(archive):
        print(f"{archive} est incomplet ou corrompu : suppression.")
        archive.unlink()

    if not fichier_valide("train.csv"):
        from getpass import getpass

        # Version récente de l'outil Kaggle : elle comprend les nouveaux jetons API
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)

        jeton = getpass("Colle ton jeton API Kaggle (laisse vide pour téléverser kaggle.json à la place) : ").strip()
        if jeton:
            os.environ["KAGGLE_API_TOKEN"] = jeton
        else:
            from google.colab import files

            print("Téléverse ton fichier kaggle.json")
            files.upload()
            dossier_kaggle = Path.home() / ".kaggle"
            dossier_kaggle.mkdir(exist_ok=True)
            Path("kaggle.json").replace(dossier_kaggle / "kaggle.json")
            os.chmod(dossier_kaggle / "kaggle.json", 0o600)

        subprocess.run(["kaggle", "competitions", "download",
                        "-c", "new-york-city-taxi-fare-prediction", "-f", "train.csv"], check=True)

    print("Fichiers prêts :", sorted(p.name for p in Path(".").glob("*.csv*")))

## 1. Les bibliothèques

In [ ]:
# Hyperopt n'est pas toujours installé : décommente la ligne suivante si l'import échoue
# %pip install -q hyperopt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

## 2. Charger et nettoyer les données

Paramètres : le nombre de lignes lues, et le nombre d'essais d'Hyperopt (section 8).

In [ ]:
NB_LIGNES = 3_000_000
MAX_EVALS = 20

# Le fichier téléchargé depuis Kaggle s'appelle train.csv.zip : Pandas le lit sans le décompresser
FICHIER_TRAIN = "train.csv" if Path("train.csv").exists() else "train.csv.zip"

In [ ]:
train = pd.read_csv(FICHIER_TRAIN, nrows=NB_LIGNES)
print("Dimensions du jeu d'entraînement :", train.shape)

In [ ]:
LIMITES_NY = {"min_lng": -74.263242, "min_lat": 40.573143,
              "max_lng": -72.986532, "max_lat": 41.709555}


def dans_zone(lat, lng, zone):
    """True si le point (lat, lng) est dans le rectangle `zone` (fonctionne sur des colonnes entières)."""
    return ((lat >= zone["min_lat"]) & (lat <= zone["max_lat"]) &
            (lng >= zone["min_lng"]) & (lng <= zone["max_lng"]))


def preparer(df):
    """Variables de date et suppression des courses aberrantes."""
    df = df.copy()
    dates = pd.to_datetime(df["pickup_datetime"], format="%Y-%m-%d %H:%M:%S UTC")
    df["pickup_day"] = dates.dt.day
    df["pickup_hour"] = dates.dt.hour
    df["pickup_day_of_week"] = (dates.dt.dayofweek + 1) % 7  # dimanche = 0, ..., samedi = 6
    df["pickup_month"] = dates.dt.month
    df["pickup_year"] = dates.dt.year

    dans_ny = (dans_zone(df["pickup_latitude"], df["pickup_longitude"], LIMITES_NY)
               & dans_zone(df["dropoff_latitude"], df["dropoff_longitude"], LIMITES_NY))
    return df[(df["fare_amount"] >= 0) & dans_ny & (df["passenger_count"] <= 8)]


train = preparer(train)
print("Dimensions après nettoyage :", train.shape)

## 3. Séparer l'entraînement et la validation

In [ ]:
cible = "fare_amount"
X = train.drop(columns=["key", "pickup_datetime", cible])
y = train[cible]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=123)
print("Entraînement :", X_train.shape)
print("Validation :", X_valid.shape)

## 4. Le modèle de référence

On prédit le prix moyen pour toutes les courses : c'est le chiffre à battre.

In [ ]:
prix_moyen = round(y_train.mean(), 2)
print("Prix moyen d'une course :", prix_moyen)

pred_reference = np.full(len(y_valid), prix_moyen)
rmse_reference = root_mean_squared_error(y_valid, pred_reference)
print(f"RMSE de référence sur la validation : {rmse_reference:.4f}")

## 5. Quatre modèles sans ingénierie des caractéristiques

### Régression linéaire

In [ ]:
resultats = []


def enregistrer(nom, rmse_entrainement, rmse_validation):
    """Garde les erreurs d'un modèle et les affiche."""
    ecart = abs(rmse_entrainement - rmse_validation)
    resultats.append({"modele": nom, "rmse_entrainement": rmse_entrainement,
                      "rmse_validation": rmse_validation, "ecart": ecart})
    print(f"{nom} : entraînement {rmse_entrainement:.4f} | validation {rmse_validation:.4f} | écart {ecart:.4f}")


lm = LinearRegression()
lm.fit(X_train, y_train)

enregistrer("Régression linéaire",
            root_mean_squared_error(y_train, lm.predict(X_train)),
            root_mean_squared_error(y_valid, lm.predict(X_valid)))

### Random Forest

Plusieurs minutes et beaucoup de mémoire sur 3 millions de lignes.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=883, n_jobs=-1)
rf.fit(X_train, y_train)

enregistrer("Random Forest",
            root_mean_squared_error(y_train, rf.predict(X_train)),
            root_mean_squared_error(y_valid, rf.predict(X_valid)))

### LightGBM

Le nombre d'arbres est choisi par validation croisée (10 plis) avec arrêt anticipé.

In [ ]:
def entrainer_lightgbm(X_train, y_train, nfold=5):
    """Choisit le nombre d'arbres par validation croisée, puis entraîne le modèle sur tout le jeu."""
    params = {"objective": "regression", "metric": "rmse", "num_leaves": 31, "verbose": -1}
    donnees = lgb.Dataset(X_train, label=y_train)
    cv = lgb.cv(params, donnees, num_boost_round=5000, nfold=nfold, stratified=False,
                callbacks=[lgb.early_stopping(20, verbose=False)])
    nb_arbres = len(cv["valid rmse-mean"])
    return lgb.train(params, donnees, num_boost_round=nb_arbres), nb_arbres


modele_lgb, nb_arbres = entrainer_lightgbm(X_train, y_train, nfold=10)
print("Nombre d'arbres retenu :", nb_arbres)

enregistrer("LightGBM",
            root_mean_squared_error(y_train, modele_lgb.predict(X_train)),
            root_mean_squared_error(y_valid, modele_lgb.predict(X_valid)))

### XGBoost

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
params_xgb = {"objective": "reg:squarederror", "eval_metric": "rmse"}

cv_xgb = xgb.cv(params_xgb, dtrain, num_boost_round=5000, nfold=5, early_stopping_rounds=20)
modele_xgb = xgb.train(params_xgb, dtrain, num_boost_round=len(cv_xgb))

enregistrer("XGBoost",
            root_mean_squared_error(y_train, modele_xgb.predict(dtrain)),
            root_mean_squared_error(y_valid, modele_xgb.predict(dvalid)))

### Comparaison

In [ ]:
tableau = pd.DataFrame(resultats).round(4)
print(tableau.to_string(index=False))

## 6. Ingénierie des caractéristiques

Les variables de la partie 1 : aéroports, quartiers, bas de Manhattan, distances aux centres des aéroports et des quartiers.

In [ ]:
aeroports = {
    "JFK": {"min_lng": -73.8352, "min_lat": 40.6195, "max_lng": -73.7401, "max_lat": 40.6659},
    "EWR": {"min_lng": -74.1925, "min_lat": 40.6700, "max_lng": -74.1531, "max_lat": 40.7081},
    "la_guardia": {"min_lng": -73.8895, "min_lat": 40.7664, "max_lng": -73.8550, "max_lat": 40.7931},
}
quartiers = {
    "manhattan": {"min_lng": -74.0479, "min_lat": 40.6829, "max_lng": -73.9067, "max_lat": 40.8820},
    "queens": {"min_lng": -73.9630, "min_lat": 40.5431, "max_lng": -73.7004, "max_lat": 40.8007},
    "brooklyn": {"min_lng": -74.0421, "min_lat": 40.5707, "max_lng": -73.8334, "max_lat": 40.7395},
    "bronx": {"min_lng": -73.9339, "min_lat": 40.7855, "max_lng": -73.7654, "max_lat": 40.9176},
    "staten_island": {"min_lng": -74.2558, "min_lat": 40.4960, "max_lng": -74.0522, "max_lat": 40.6490},
}
bas_manhattan = {"min_lng": -74.0194, "min_lat": 40.6997, "max_lng": -73.9716, "max_lat": 40.7427}

# Centres (longitude, latitude) des aéroports et des quartiers
centres = {
    "jfk": (-73.7900, 40.6437), "ewr": (-74.1843, 40.6924), "laguardia": (-73.8733, 40.7746),
    "manhattan": (-73.9664, 40.7909), "queens": (-73.8317, 40.7038), "brooklyn": (-73.9489, 40.6551),
    "bronx": (-73.8568, 40.8572), "statenisland": (-74.1540, 40.5725),
}

In [ ]:
def distance_miles(lat1, lng1, lat2, lng2):
    """Distance de Haversine en miles (fonctionne sur des colonnes entières)."""
    p = np.pi / 180
    a = (0.5 - np.cos((lat2 - lat1) * p) / 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lng2 - lng1) * p)) / 2)
    return 0.6213712 * 12742 * np.arcsin(np.sqrt(a))


NOMS_QUARTIERS = [*quartiers, "others"]


def ajouter_caracteristiques(df):
    df = df.copy()
    for sens in ("pickup", "dropoff"):
        lat, lng = df[f"{sens}_latitude"], df[f"{sens}_longitude"]

        for nom, zone in aeroports.items():
            df[f"is_{sens}_{nom}"] = dans_zone(lat, lng, zone).astype(int)
        df[f"is_{sens}_lower_manhattan"] = dans_zone(lat, lng, bas_manhattan).astype(int)

        for nom, (lng_centre, lat_centre) in centres.items():
            df[f"{sens}_distance_{nom}"] = distance_miles(lat, lng, lat_centre, lng_centre)

        # Premier rectangle qui contient le point (les rectangles se chevauchent)
        quartier = np.select([dans_zone(lat, lng, zone) for zone in quartiers.values()],
                             list(quartiers), default="others")
        df[f"{sens}_borough"] = pd.Categorical(quartier, categories=NOMS_QUARTIERS)

    # Encodage "one-hot" : une colonne 0/1 par quartier
    return pd.get_dummies(df, columns=["pickup_borough", "dropoff_borough"], dtype=int)


X_train = ajouter_caracteristiques(X_train)
X_valid = ajouter_caracteristiques(X_valid)
print("Nombre de variables :", X_train.shape[1])

In [ ]:
modele_lgb_fe, nb_arbres_fe = entrainer_lightgbm(X_train, y_train, nfold=5)
print("Nombre d'arbres retenu :", nb_arbres_fe)

enregistrer("LightGBM + variables de la partie 1",
            root_mean_squared_error(y_train, modele_lgb_fe.predict(X_train)),
            root_mean_squared_error(y_valid, modele_lgb_fe.predict(X_valid)))

## 7. Régler le modèle avec Hyperopt

Chaque essai entraîne un modèle complet, d'où le nombre d'essais limité à 20 (`MAX_EVALS`, section 2). Avec 100 essais, le résultat est un peu mieux optimisé, mais le gain est faible pour près de 2 heures de calcul. La graine `rstate` rend la recherche reproductible : tu retrouves les mêmes valeurs en relançant le notebook.

In [ ]:
def objectif(espace):
    """Entraîne un LightGBM avec ces hyperparamètres et renvoie son RMSE de validation."""
    modele = lgb.LGBMRegressor(
        objective="regression", boosting_type="gbdt", num_leaves=60, n_estimators=5000,
        subsample_freq=20, max_depth=int(espace["max_depth"]), subsample=espace["subsample"],
        colsample_bytree=espace["colsample"], n_jobs=-1, verbose=-1)
    modele.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="rmse",
               callbacks=[lgb.early_stopping(20, verbose=False)])
    rmse = root_mean_squared_error(y_valid, modele.predict(X_valid))
    return {"loss": rmse, "status": STATUS_OK}


espace = {
    "max_depth": hp.quniform("x_max_depth", 5, 30, 3),
    "subsample": hp.uniform("x_subsample", 0.8, 1),
    "colsample": hp.uniform("x_colsample", 0.3, 1),
}

essais = Trials()
meilleurs = fmin(fn=objectif, space=espace, algo=tpe.suggest, max_evals=MAX_EVALS,
                 trials=essais, rstate=np.random.default_rng(1234), show_progressbar=False)

meilleurs = {cle: float(meilleurs[cle]) for cle in ("x_max_depth", "x_subsample", "x_colsample")}
print(meilleurs)

In [ ]:
modele_final = lgb.LGBMRegressor(
    objective="regression", boosting_type="gbdt", num_leaves=60, n_estimators=5000,
    subsample_freq=20, max_depth=int(meilleurs["x_max_depth"]), subsample=meilleurs["x_subsample"],
    colsample_bytree=meilleurs["x_colsample"], n_jobs=-1, verbose=-1)
modele_final.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="rmse",
                 callbacks=[lgb.early_stopping(20, verbose=False)])
print("Nombre d'arbres retenu :", modele_final.best_iteration_)

enregistrer("LightGBM réglé",
            root_mean_squared_error(y_train, modele_final.predict(X_train)),
            root_mean_squared_error(y_valid, modele_final.predict(X_valid)))

## 8. Quelles variables comptent le plus ?

In [ ]:
importances = pd.Series(modele_final.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 15))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index,
            palette="husl", legend=False)
Path("images").mkdir(exist_ok=True)
plt.savefig("images/taxi-importance-variables-lightgbm.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Piste : ajouter la distance du trajet

Les modèles ci-dessus n'utilisent pas `trip_distance`, créée dans la partie 1. Cette cellule l'ajoute et mesure l'effet sur le RMSE de validation.

In [ ]:
for jeu in (X_train, X_valid):
    jeu["trip_distance"] = distance_miles(jeu["pickup_latitude"], jeu["pickup_longitude"],
                                          jeu["dropoff_latitude"], jeu["dropoff_longitude"])

modele_distance, nb_arbres_distance = entrainer_lightgbm(X_train, y_train, nfold=5)
print("Nombre d'arbres retenu :", nb_arbres_distance)

enregistrer("LightGBM + distance du trajet",
            root_mean_squared_error(y_train, modele_distance.predict(X_train)),
            root_mean_squared_error(y_valid, modele_distance.predict(X_valid)))

In [ ]:
tableau = pd.DataFrame(resultats).round(4)
print(tableau.to_string(index=False))

## 10. Récupérer le graphique (Google Colab)

Le graphique d'importance est enregistré dans le dossier `images/`. Dans Colab, cette cellule le télécharge sur ton ordinateur.

In [ ]:
if EN_COLAB:
    from google.colab import files
    files.download("images/taxi-importance-variables-lightgbm.png")